  ##  Importing libraries

In [1]:
## importing libraries
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

PROJECT_DIR = Path.cwd()
FILE_DIR = PROJECT_DIR / "Data_Collection"
FILE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "Production_Crops_Livestock_E_All_Data.csv"
if not DATA_PATH.exists():
    # Accept the common browser-upload suffix, or another single matching source file.
    candidates = sorted(PROJECT_DIR.glob("Production_Crops_Livestock_E_All_Data*.csv"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            "Dataset not found. Put Production_Crops_Livestock_E_All_Data.csv "
            "in the same folder as this notebook."
        )

print("Dataset:", DATA_PATH)
print("FILE_PATH:", FILE_DIR)


Dataset: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Production_Crops_Livestock_E_All_Data.csv
FILE_PATH: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection


  ## loading the original dataset

In [2]:
## loading the original dataset
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
print("Original shape:", raw_df.shape)
display(raw_df.head())
print("Columns:", raw_df.columns.tolist()[:12], "...")


Original shape: (79711, 198)


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Unit,Y1961,Y1961F,Y1961N,Y1962,Y1962F,Y1962N,Y1963,Y1963F,Y1963N,Y1964,Y1964F,Y1964N,Y1965,Y1965F,Y1965N,Y1966,Y1966F,Y1966N,Y1967,Y1967F,Y1967N,Y1968,Y1968F,Y1968N,Y1969,Y1969F,Y1969N,Y1970,Y1970F,Y1970N,Y1971,Y1971F,Y1971N,Y1972,Y1972F,Y1972N,Y1973,Y1973F,Y1973N,Y1974,Y1974F,...,Y2007F,Y2007N,Y2008,Y2008F,Y2008N,Y2009,Y2009F,Y2009N,Y2010,Y2010F,Y2010N,Y2011,Y2011F,Y2011N,Y2012,Y2012F,Y2012N,Y2013,Y2013F,Y2013N,Y2014,Y2014F,Y2014N,Y2015,Y2015F,Y2015N,Y2016,Y2016F,Y2016N,Y2017,Y2017F,Y2017N,Y2018,Y2018F,Y2018N,Y2019,Y2019F,Y2019N,Y2020,Y2020F,Y2020N,Y2021,Y2021F,Y2021N,Y2022,Y2022F,Y2022N,Y2023,Y2023F,Y2023N
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,ha,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,...,A,NaN,12000.0,A,NaN,11029.0,A,NaN,11210.0,A,NaN,13469.0,A,NaN,13490.0,A,NaN,14114.0,A,NaN,13703.0,A,NaN,14676.0,A,NaN,19481.0,A,NaN,19793.0,A,NaN,20053.0,A,NaN,29203.0,A,NaN,22134.0,A,NaN,36862.0,A,NaN,36462.0,A,NaN,37000.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5412,Yield,kg/ha,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,NaN,3500.0,A,NaN,3915.4,E,NaN,4995.5,A,NaN,4500.0,A,NaN,4596.0,A,NaN,2991.0,A,NaN,1999.6,A,NaN,1652.1,A,NaN,1685.9,A,NaN,1378.8,A,NaN,1716.1,A,NaN,1308.3,A,NaN,1775.9,A,NaN,1743.2,A,NaN,1742.0,A,NaN,1810.8,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,t,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,NaN,0.0,A,...,A,NaN,42000.0,A,NaN,43183.0,X,Unofficial figure,56000.0,A,NaN,60611.0,A,NaN,62000.0,A,NaN,42215.0,A,NaN,27400.0,A,NaN,24246.0,A,NaN,32843.0,A,NaN,27291.0,A,NaN,34413.0,A,NaN,38205.0,A,NaN,39307.0,A,NaN,64256.0,A,NaN,63515.0,A,NaN,67000.0,A,NaN
3,2,'004,Afghanistan,711,'01654,"Anise, badian, coriander, cumin, caraway, fenn...",5312,Area harvested,ha,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,NaN,NaN,M,...,E,NaN,15000.0,E,NaN,17748.0,I,NaN,17000.0,E,NaN,19500.0,I,NaN,18500.0,E,NaN,18500.0,E,NaN,30000.0,E,NaN,25000.0,E,NaN,24500.0,E,NaN,26160.0,I,NaN,25220.0,E,NaN,25293.0,E,NaN,25558.0,E,NaN,25357.0,E,NaN,25403.0,E,NaN,25439.0,E,NaN
4,2,'004,Afghanistan,711,'01654,"Anise, badian, coriander, cumin, caraway, fenn...",5412,Yield,kg/ha,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,NaN,600.0,E,NaN,620.3,E,NaN,600.0,E,NaN,641.4,E,NaN,675.7,E,NaN,675.7,E,NaN,716.7,E,NaN,720.0,E,NaN,707.5,E,NaN,697.0,E,NaN,708.0,E,NaN,704.1,E,NaN,703.0,E,NaN,705.0,E,NaN,704.0,E,NaN,704.0,E,NaN


Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Unit', 'Y1961', 'Y1961F', 'Y1961N'] ...


  ## identifying year columns and required identifier columns

In [3]:

## identifying year columns and required identifier columns

import re

id_cols = ["Area Code", "Area", "Item Code", "Item", "Element", "Unit"]

# Check required identifier columns
missing_id_cols = [col for col in id_cols if col not in raw_df.columns]

if missing_id_cols:
    raise ValueError(
        f"Required identifier columns are missing: {missing_id_cols}"
    )

# Clean column names in case they contain extra spaces
raw_df.columns = raw_df.columns.astype(str).str.strip()

# Identify annual columns such as Y1961, Y2022, Y2023
year_cols = [
    col for col in raw_df.columns
    if re.fullmatch(r"Y\d{4}", col)
]

if not year_cols:
    print("First 30 column names:")
    print(raw_df.columns[:30].tolist())

    raise ValueError(
        "No annual columns matching Y#### were found. "
        "Check the actual column names in raw_df."
    )

# Sort annual columns chronologically
year_cols = sorted(year_cols, key=lambda col: int(col[1:]))

print(
    f"Year columns found: {len(year_cols)} | "
    f"Range: {year_cols[0]} to {year_cols[-1]}"
)


Year columns found: 63 | Range: Y1961 to Y2023


  ## selecting yield records and reshaping wide data into long format

In [4]:
## selecting yield records and reshaping wide data into long format
# The source file contains multiple elements. Keep only rows whose Element is Yield.
yield_df = raw_df.loc[raw_df["Element"].astype(str).str.strip().str.casefold().eq("yield")].copy()
if yield_df.empty:
    raise ValueError("No rows with Element == 'Yield' were found. Inspect raw_df['Element'].unique().")

yield_long = yield_df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="Year",
    value_name="Yield_kg_ha"
)
yield_long["Year"] = yield_long["Year"].str[1:].astype(int)
yield_long["Yield_kg_ha"] = pd.to_numeric(yield_long["Yield_kg_ha"], errors="coerce")
yield_long = yield_long.dropna(subset=["Yield_kg_ha"]).copy()
yield_long = yield_long.sort_values(["Area Code", "Item Code", "Year"]).reset_index(drop=True)

print("Long-format shape:", yield_long.shape)
display(yield_long.head())
print("Year range:", yield_long["Year"].min(), "-", yield_long["Year"].max())
print("Units:", yield_long["Unit"].value_counts(dropna=False).head(10).to_dict())


Long-format shape: (940949, 8)


,Area Code,Area,Item Code,Item,Element,Unit,Year,Yield_kg_ha
0,1,Armenia,15,Wheat,Yield,kg/ha,1992,2160.0
1,1,Armenia,15,Wheat,Yield,kg/ha,1993,2225.7
2,1,Armenia,15,Wheat,Yield,kg/ha,1994,1784.2
3,1,Armenia,15,Wheat,Yield,kg/ha,1995,2548.0
4,1,Armenia,15,Wheat,Yield,kg/ha,1996,2356.9


Year range: 1961 - 2023
Units: {'kg/ha': 808847, '100 g/An': 85309, '100 mg/An': 30244, 'No/An': 16549}


  ## validating records and inspecting potential data-quality issues

In [5]:
## validating records and inspecting potential data-quality issues
key_cols = ["Area Code", "Item Code", "Year"]
print("Duplicate area-item-year records:", yield_long.duplicated(key_cols).sum())
print("Missing values by column:")
display(yield_long.isna().sum().to_frame("missing_count"))
print("Negative yields:", int((yield_long["Yield_kg_ha"] < 0).sum()))
print("Zero yields:", int((yield_long["Yield_kg_ha"] == 0).sum()))
print("Largest yields (review; do not automatically delete):")
display(yield_long.nlargest(15, "Yield_kg_ha")[["Area", "Item", "Year", "Unit", "Yield_kg_ha"]])


Duplicate area-item-year records: 16502
Missing values by column:


,missing_count
Area Code,0
Area,0
Item Code,0
Item,0
Element,0
Unit,0
Year,0
Yield_kg_ha,0


Negative yields: 0
Zero yields: 364
Largest yields (review; do not automatically delete):


,Area,Item,Year,Unit,Yield_kg_ha
802125,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1988,100 mg/An,80109661.0
802127,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1989,100 mg/An,69829810.0
802123,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1987,100 mg/An,68350565.0
802121,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1986,100 mg/An,65065714.0
802119,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1985,100 mg/An,63890698.0
802137,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1994,100 mg/An,62949895.0
802073,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1962,100 mg/An,60855192.0
802117,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1984,100 mg/An,54178788.0
802077,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1964,100 mg/An,53208387.0
802139,Eastern Asia,"Eggs from other birds in shell, fresh, n.e.c.",1995,100 mg/An,52143385.0


  ## selecting the project crops

In [6]:
## selecting the project crops
selected_crops = ["Maize (corn)", "Rice", "Wheat"]
crop_subset = yield_long.loc[yield_long["Item"].isin(selected_crops)].copy()
if crop_subset.empty:
    raise ValueError(f"None of the selected crops were found: {selected_crops}")

print("Rows for selected crops:", len(crop_subset))
display(crop_subset.groupby("Item")["Yield_kg_ha"].agg(["count", "min", "median", "mean", "max"]).round(2))


Rows for selected crops: 29685


,count,min,median,mean,max
Item,,,,,
Maize (corn),11566,0.1,1929.65,3153.34,39405.7
Rice,9187,96.8,2852.20,3273.72,11351.1
Wheat,8932,0.0,2028.40,2516.96,10667.7


  ## removing reviewed aggregate geographic areas

In [7]:
## removing reviewed aggregate geographic areas
# This is an explicit reviewed list, not a universal country classifier.
# Country names such as South Africa and Micronesia (Federated States of) are intentionally retained.
aggregate_areas = [
    "Africa", "Americas", "Asia", "Caribbean", "Central America", "Central Asia",
    "Eastern Africa", "Eastern Asia", "Eastern Europe", "Europe",
    "Land Locked Developing Countries", "Least Developed Countries",
    "Low Income Food Deficit Countries", "Melanesia", "Micronesia", "Middle Africa",
    "Net Food Importing Developing Countries", "Northern Africa", "Northern Europe",
    "Oceania", "Small Island Developing States", "South-eastern Asia",
    "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa",
    "Western Asia", "Western Europe", "World"
]
country_crop_data = crop_subset.loc[~crop_subset["Area"].isin(aggregate_areas)].copy()
country_crop_data = country_crop_data.sort_values(["Area Code", "Item Code", "Year"]).reset_index(drop=True)

print("Before aggregate-area filter:", len(crop_subset))
print("After aggregate-area filter:", len(country_crop_data))
print("Areas retained:", country_crop_data["Area"].nunique())
print("Duplicate area-item-year records:", country_crop_data.duplicated(key_cols).sum())
display(country_crop_data.groupby("Item").size().rename("rows").to_frame())


Before aggregate-area filter: 29685
After aggregate-area filter: 24588
Areas retained: 196
Duplicate area-item-year records: 0


,rows
Item,
Maize (corn),9820
Rice,7488
Wheat,7280


  ## saving cleaned datasets for the next notebooks

In [8]:
## saving cleaned datasets for the next notebooks
yield_long_path = FILE_DIR / "yield_long.csv"
country_crop_path = FILE_DIR / "country_crop_data.csv"

yield_long.to_csv(yield_long_path, index=False)
country_crop_data.to_csv(country_crop_path, index=False)

print("Saved:", yield_long_path)
print("Saved:", country_crop_path)
print("Next: run 02_Univariate_and_Bivariate_Analysis.ipynb")


Saved: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\yield_long.csv
Saved: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\country_crop_data.csv
Next: run 02_Univariate_and_Bivariate_Analysis.ipynb
